# Phase 3 — Retention & cohorts

**Questions:** *Who* churns and *when*?

| Output | Source model | Population |
|---|---|---|
| Cohort retention heatmap + curves | `mart_cohort_retention` | New customers only (registered ≥ 2015-01-01) — complete data |
| Monthly renewal churn | `mart_monthly_kpis` | All subscription periods — complete data |
| Churn rate by plan / payment / auto-renew / price | `mart_churn_by_segment` | KKBox label population (memberships expiring Mar 2017) — complete data |

**Definitions used everywhere**
* *Active subscriber (month m)* = a subscription period covers the last day of month m.
* *Retention at month k* = share of a signup cohort active at the end of month k after signup.
* *Churn (KKBox rule)* = no new valid subscription within 30 days of the membership ending.

No listening logs are used here, so nothing in this notebook is affected by sampling.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from streampulse.db import query, t, save_result, banner, RESULTS_DIR, IS_FIXTURE

banner()
FIG = RESULTS_DIR / "figures"
FIG.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

# Segments smaller than this are shown but never quoted as findings (too noisy).
MIN_SEGMENT_USERS = 5 if IS_FIXTURE else 1000

## 3.1 Cohort retention heatmap
Counts are summed across first-plan groups before dividing, so the rate is weighted correctly (never average the rates).

In [ ]:
coh = query(f"select * from {t('mart_cohort_retention')}")
coh["cohort_month"] = pd.to_datetime(coh["cohort_month"])

cohort = (coh.groupby(["cohort_month", "months_since_signup"], as_index=False)[["cohort_size", "active_users"]]
             .sum())
cohort["retention"] = cohort["active_users"] / cohort["cohort_size"]
heat = cohort[cohort.months_since_signup <= 12].pivot(index="cohort_month", columns="months_since_signup", values="retention")

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(heat.values, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_yticks(range(len(heat.index)), [d.strftime("%Y-%m") for d in heat.index], fontsize=7)
ax.set_xticks(range(heat.shape[1]), heat.columns)
ax.set_xlabel("Months since signup"); ax.set_ylabel("Signup cohort")
ax.set_title("Share of each signup cohort still subscribed at month-end")
fig.colorbar(im, ax=ax, label="retention")
fig.savefig(FIG / "p03_cohort_heatmap.png", dpi=150, bbox_inches="tight")
heat.round(3).head(12)

## 3.2 Average retention curve
Only cohorts with **12 full months observed** are averaged. Otherwise month 12 would be computed from old cohorts only while month 1 includes young ones — a mix shift, not a real curve.

In [ ]:
last_month = coh["calendar_month"].max()
full = coh[pd.to_datetime(coh["cohort_month"]) <= pd.to_datetime(last_month) - pd.DateOffset(months=12)]
full = full[full.months_since_signup <= 12]

curve = full.groupby("months_since_signup")[["active_users", "cohort_size"]].sum()
curve["retention"] = curve.active_users / curve.cohort_size

by_plan = full.groupby(["first_plan_group", "months_since_signup"])[["active_users", "cohort_size"]].sum()
by_plan["retention"] = by_plan.active_users / by_plan.cohort_size
by_plan = by_plan["retention"].unstack(0)
plan_sizes = full[full.months_since_signup == 0].groupby("first_plan_group")["cohort_size"].sum()

fig, ax = plt.subplots()
ax.plot(curve.index, curve.retention, color="black", lw=2.5, label="All new customers")
for g in by_plan.columns:
    ax.plot(by_plan.index, by_plan[g], lw=1.2, label=f"{g} (n={plan_sizes.get(g, 0):,})")
ax.set_ylim(0, 1); ax.set_xlabel("Months since signup"); ax.set_ylabel("Retention")
ax.set_title("Retention curve by first plan (cohorts with 12 months observed)")
ax.legend(fontsize=8)
fig.savefig(FIG / "p03_retention_curves.png", dpi=150, bbox_inches="tight")
print("cohorts used:", full.cohort_month.nunique(), " customers:", int(curve.loc[0, "cohort_size"]))
curve.round(4)

## 3.3 When do people churn? Monthly renewal churn
Warm-up months (first 6) and months whose 30-day churn window runs past the data end are excluded.

In [ ]:
kpi = query(f"select * from {t('mart_monthly_kpis')} order by month_start")
kpi["month_start"] = pd.to_datetime(kpi["month_start"])
rc = kpi[(~kpi.is_warmup_month) & (~kpi.is_churn_unobservable)].copy()

fig, ax = plt.subplots()
ax.plot(rc.month_start, rc.renewal_churn_rate, marker="o")
ax.set_ylim(0, None); ax.set_ylabel("Share of ending periods not renewed within 30 days")
ax.set_title("Monthly renewal churn")
fig.savefig(FIG / "p03_renewal_churn.png", dpi=150, bbox_inches="tight")
rc[["month_start", "periods_ending", "periods_churned", "renewal_churn_rate"]]

## 3.4 Who churns? Churn rate by segment (KKBox label population)
95% Wilson confidence intervals: a segment's rate is only different from another's if the intervals don't overlap much — small segments have wide intervals.

In [ ]:
seg = query(f"select * from {t('mart_churn_by_segment')}")
lo, hi = proportion_confint(seg.churners, seg.users, alpha=0.05, method="wilson")
seg["ci_low"], seg["ci_high"] = lo, hi
seg["quotable"] = seg.users >= MIN_SEGMENT_USERS

overall_users = int(seg[seg.dimension == "auto_renew"].users.sum())
overall_churners = int(seg[seg.dimension == "auto_renew"].churners.sum())
overall_rate = overall_churners / overall_users
print(f"Label population: {overall_users:,} members, churn rate {overall_rate:.2%}")

for dim in ["plan_length", "auto_renew", "monthly_price", "payment_method", "discounted", "cancelled_before", "tenure", "age"]:
    d = seg[seg.dimension == dim].sort_values("churn_rate", ascending=False)
    display(d[["segment", "users", "share_of_users", "churn_rate", "ci_low", "ci_high", "churn_lift_vs_overall", "quotable"]]
            .style.set_caption(dim).format({"share_of_users": "{:.1%}", "churn_rate": "{:.2%}", "ci_low": "{:.2%}",
                                            "ci_high": "{:.2%}", "churn_lift_vs_overall": "{:.2f}x", "users": "{:,}"}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, dim in zip(axes, ["auto_renew", "plan_length", "monthly_price"]):
    d = seg[(seg.dimension == dim) & seg.quotable].sort_values("segment")
    err = np.clip(np.vstack([d.churn_rate - d.ci_low, d.ci_high - d.churn_rate]).astype(float), 0, None)
    ax.barh(d.segment, d.churn_rate, xerr=err, color="#4C78A8")
    ax.axvline(overall_rate, color="grey", ls="--", lw=1)
    ax.set_title(dim); ax.set_xlabel("churn rate")
fig.tight_layout()
fig.savefig(FIG / "p03_churn_by_segment.png", dpi=150, bbox_inches="tight")

## 3.5 Is the auto-renew gap real? Two-proportion z-test
H0: churn rate is the same with auto-renew on and off.

In [ ]:
ar = seg[seg.dimension == "auto_renew"].set_index("segment")
z, p = proportions_ztest(ar.churners.values, ar.users.values)
diff = ar.loc["auto-renew off", "churn_rate"] - ar.loc["auto-renew on", "churn_rate"]
print(f"auto-renew off minus on: {diff:+.2%} points, z={z:.2f}, p={p:.3g}")
print("Note: this is an association, not a cause. Auto-renew users may differ in other ways (Phase 5 controls for some).")

In [ ]:
quotable = seg[seg.quotable]
res = {
    "label_population_users": overall_users,
    "label_churn_rate": overall_rate,
    "new_customer_cohorts_used_for_curve": int(full.cohort_month.nunique()),
    "new_customers_in_curve": int(curve.loc[0, "cohort_size"]),
    "avg_retention": {f"month_{k}": float(curve.loc[k, "retention"]) for k in [0, 1, 3, 6, 12] if k in curve.index},
    "retention_month_6_by_first_plan": {g: float(by_plan.loc[6, g]) for g in by_plan.columns if 6 in by_plan.index},
    "renewal_churn_rate": {"mean": float(rc.renewal_churn_rate.mean()), "min": float(rc.renewal_churn_rate.min()),
                           "max": float(rc.renewal_churn_rate.max()),
                           "max_month": str(rc.loc[rc.renewal_churn_rate.idxmax(), "month_start"].date())},
    "churn_by_auto_renew": ar[["users", "churn_rate"]].to_dict("index"),
    "auto_renew_ztest": {"diff_off_minus_on": float(diff), "z": float(z), "p_value": float(p)},
    "highest_churn_segments": quotable.sort_values("churn_rate", ascending=False).head(5)[
        ["dimension", "segment", "users", "churn_rate", "churn_lift_vs_overall"]].to_dict("records"),
    "lowest_churn_segments": quotable.sort_values("churn_rate").head(3)[
        ["dimension", "segment", "users", "churn_rate"]].to_dict("records"),
}
save_result("phase03_retention", res)
res